# Tune CNN: learning_rate x batch_size

10-trial random search (Optuna `RandomSampler(seed=42)`, matching the SVM's
`svm-tuning.ipynb` search style). Architecture frozen (Table 3.5). Split and
init are fixed — every trial reuses `cnn-latest/results/splits/clean-seed42.csv`
and seed 42, so trials differ only by hyperparameters.

This pass searches only `learning_rate` and `batch_size`. EarlyStopping
patience (6), ReduceLROnPlateau patience (2) and factor (0.1) stay fixed at
their current values — `run_tune.py` accepts them as CLI args for a later pass.

Selection metric: `mean_ood` = mean(A1, A2, B1, B2), computed directly each
trial by `run_tune.py`. This is explicit selection on the reported
generalization scenarios, not a held-out dev set — state that in any writeup.

Each trial is an isolated OS process (`run_tune.py`), same pattern as
`cnn-latest/train-variants.ipynb`.

## Config

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd()
if HERE.name != "cnn-revised":
    HERE = Path("training/notebooks/cnn-revised").resolve()
TRAINING_ROOT = HERE.parents[1]
TRIALS_CSV = HERE / "results" / "tuning_trials.csv"
SPLIT_CSV = HERE.parent / "cnn-latest" / "results" / "splits" / "clean-seed42.csv"
N_TRIALS = 10

cfg = pd.DataFrame([
    {"key": "here", "value": str(HERE)},
    {"key": "features", "value": str(TRAINING_ROOT / "features" / "training-clean.npz")},
    {"key": "split (fixed, reused)", "value": str(SPLIT_CSV)},
    {"key": "n_trials", "value": N_TRIALS},
    {"key": "search space", "value": "learning_rate (log-uniform 1e-4..3e-3), batch_size {16,32,64}"},
    {"key": "fixed", "value": "es_patience=6, rlr_patience=2, rlr_factor=0.1, seed=42"},
])
display(cfg)
assert SPLIT_CSV.is_file(), SPLIT_CSV
assert (HERE / "run_tune.py").is_file()

,key,value
0,here,/home/seya/code/chord-detection/training/noteb...
1,features,/home/seya/code/chord-detection/training/featu...
2,"split (fixed, reused)",/home/seya/code/chord-detection/training/noteb...
3,n_trials,10
4,search space,"learning_rate (log-uniform 1e-4..3e-3), batch_..."
5,fixed,"es_patience=6, rlr_patience=2, rlr_factor=0.1,..."


## Optuna study — random search, 10 trials

Each `ask()` gives `(lr, batch_size)`; the objective dispatches `run_tune.py` as
a subprocess (isolated OS process, GPU memory released on exit), then reads
`mean_ood` back from the row that trial just appended to `tuning_trials.csv`.
Resume-friendly: a trial whose history CSV already exists is skipped by
`run_tune.py` itself.

In [2]:
import optuna

STUDY_DB = HERE / "results" / "tuning_study.db"
STUDY_NAME = "CNN_Tuning-RandomizedSearchCV"

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=f"sqlite:///{STUDY_DB}",
    direction="maximize",
    load_if_exists=True,
    sampler=optuna.samplers.RandomSampler(seed=42),
)


def run_trial(trial_id: int, lr: float, batch_size: int) -> float:
    hist_path = HERE / "results" / "history" / f"trial{trial_id:02d}.csv"
    if not hist_path.exists():
        result = subprocess.run(
            [
                "uv", "run", "python", "run_tune.py",
                "--trial", str(trial_id),
                "--lr", str(lr),
                "--batch-size", str(batch_size),
            ],
            cwd=HERE,
            check=False,
        )
        if result.returncode != 0:
            raise RuntimeError(f"trial {trial_id} exited {result.returncode}")
    df = pd.read_csv(TRIALS_CSV)
    row = df.loc[df.trial == trial_id].iloc[0]
    return float(row.mean_ood)


def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    return run_trial(trial.number, lr, batch_size)


n_done = len(study.trials)
display(Markdown(f"**{n_done}** / {N_TRIALS} trials already in the study."))
if n_done < N_TRIALS:
    study.optimize(objective, n_trials=N_TRIALS - n_done, gc_after_trial=True)

display(Markdown(f"Best so far: trial {study.best_trial.number}, "
                  f"mean_ood={study.best_value:.4f}, params={study.best_params}"))

[I 2026-09-03 07:07:09,948] A new study created in RDB with name: CNN_Tuning-RandomizedSearchCV


**0** / 10 trials already in the study.

I0000 00:00:1788394045.811232   21969 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


/home/seya/code/chord-detection/training/notebooks/cnn-revised/run_tune.py:221: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prev = pd.concat([prev, pd.DataFrame([row])], ignore_index=True)


Using GPU: NVIDIA GeForce RTX 2080 Ti
OK trial 0: mean_ood=1.0000 A1=1.0000 A2=1.0000 B1=1.0000 B2=1.0000


...

[I 2026-09-03 07:33:47,410] Trial 8 finished with value: 0.9998263888888888 and parameters: {'lr': 0.00012476394272569445, 'batch_size': 32}. Best is trial 0 with value: 1.0.


I0000 00:00:1788395642.045543   47056 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Using GPU: NVIDIA GeForce RTX 2080 Ti
OK trial 9: mean_ood=1.0000 A1=1.0000 A2=1.0000 B1=1.0000 B2=1.0000


[I 2026-09-03 07:35:35,482] Trial 9 finished with value: 1.0 and parameters: {'lr': 0.00028180680291847244, 'batch_size': 32}. Best is trial 0 with value: 1.0.


Best so far: trial 0, mean_ood=1.0000, params={'lr': 0.0003574712922600243, 'batch_size': 16}

## Trial results

In [3]:
trials = pd.read_csv(TRIALS_CSV).sort_values(
    ["mean_ood", "test_accuracy", "val_loss"], ascending=[False, False, True]
).reset_index(drop=True)
winner_trial = int(trials.iloc[0].trial)

show = trials.copy()
show.insert(0, "", ["*" if t == winner_trial else "" for t in show.trial])
cols = ["", "trial", "lr", "batch_size", "epochs_run", "val_loss", "test_accuracy",
        "A1", "A2", "B1", "B2", "mean_ood", "minutes"]
display(Markdown(
    f"**Winner: trial {winner_trial}** — lr={trials.iloc[0].lr:.5g}, "
    f"batch_size={int(trials.iloc[0].batch_size)}, mean_ood={trials.iloc[0].mean_ood:.4f} "
    f"(tie-break: higher test_accuracy, then lower val_loss). `*` marks it."
))
display(show[cols].round(5))

**Winner: trial 0** — lr=0.00035747, batch_size=16, mean_ood=1.0000 (tie-break: higher test_accuracy, then lower val_loss). `*` marks it.

,,trial,lr,batch_size,epochs_run,val_loss,test_accuracy,A1,A2,B1,B2,mean_ood,minutes
0,*,0,0.00036,16,18,0.00000,1.0,1.00000,1.00000,1.0,1.00000,1.00000,3.45000
1,,1,0.00017,64,11,0.00000,1.0,1.00000,1.00000,1.0,1.00000,1.00000,2.03333
2,,9,0.00028,32,9,0.00000,1.0,1.00000,1.00000,1.0,1.00000,1.00000,1.75000
3,,8,0.00012,32,12,0.00000,1.0,1.00000,1.00000,1.0,0.99931,0.99983,2.18333
4,,4,0.00028,16,13,0.00000,1.0,1.00000,1.00000,1.0,0.99792,0.99948,2.51667
5,,6,0.00047,16,18,0.00000,1.0,1.00000,0.98681,1.0,0.98194,0.99219,3.30000
6,,7,0.00075,32,18,0.00000,1.0,0.99861,0.98611,1.0,0.97569,0.99010,3.03333
7,,3,0.00170,16,29,0.00001,1.0,0.99653,0.96875,1.0,0.97917,0.98611,5.01667
8,,2,0.00077,64,10,0.00000,1.0,0.97917,0.94583,1.0,0.93125,0.96406,1.81667
9,,5,0.00080,64,18,0.00000,1.0,0.99583,0.89583,1.0,0.90278,0.94861,2.90000


## Promote the winner

Copies `weights/trial{winner}.keras` → `weights/clean-seed42.keras` and the
matching history CSV, the filename every downstream consumer
(`run_eval.py`, `onset-classify.ipynb`, `app/models` symlink) expects.

In [4]:
import shutil

src_w = HERE / "weights" / f"trial{winner_trial:02d}.keras"
dst_w = HERE / "weights" / "clean-seed42.keras"
src_h = HERE / "results" / "history" / f"trial{winner_trial:02d}.csv"
dst_h = HERE / "results" / "history" / "clean-seed42.csv"

assert src_w.is_file(), src_w
shutil.copy2(src_w, dst_w)
shutil.copy2(src_h, dst_h)

display(Markdown(
    f"Promoted trial {winner_trial} → `{dst_w.relative_to(TRAINING_ROOT)}` "
    f"({dst_w.stat().st_size / 1e6:.1f} MB)"
))

Promoted trial 0 → `notebooks/cnn-revised/weights/clean-seed42.keras` (124.2 MB)